# NBEATS Model Univariate


**NBEATS (Neural Basis Expansion Analysis for Time Series)** is a deep learning architecture specifically designed for univariate time series forecasting. Implemented using the NeuralForecast package from Nixtla, it forecasts multiple individual series independently by decomposing the time series into interpretable components (trend and seasonality) using basis functions.

NeuralForecast uses a vectorized batching approach similar to our TimeSeriesDatasetVectorizedExog, processing all 1502 series in parallel rather than sequentially. This architecture delivers significant training speed improvements comparable to our vectorized implementations, while maintaining the interpretability of decomposed forecast components.

## Architecture

```bash
Input Sequence (seq_length timesteps)
         ↓
    ┌────────────────────────────────┐
    │   Stack 1: Trend Block(s)      │
    │   - Polynomial basis functions  │
    │   - MLPs for basis coefficients │
    │   - Backcast + Forecast         │
    └────────────────────────────────┘
         ↓ (residual connection)
    ┌────────────────────────────────┐
    │   Stack 2: Seasonality Block(s)│
    │   - Fourier basis functions     │
    │   - MLPs for basis coefficients │
    │   - Backcast + Forecast         │
    └────────────────────────────────┘
         ↓ (residual connection)
    ┌────────────────────────────────┐
    │   Stack 3: Generic Block(s)    │
    │   - Learnable basis functions   │
    │   - MLPs for basis coefficients │
    │   - Backcast + Forecast         │
    └────────────────────────────────┘
         ↓
    Final Forecast (horizon)
```

## Model Structure

### Block Architecture
Each block in NBEATS consists of:
1. **Fully Connected Layers (MLP)**: Process input sequence
2. **Basis Expansion**: Decompose signal using basis functions
   - **Trend Stack**: Polynomial basis for long-term patterns
   - **Seasonality Stack**: Fourier basis (harmonics) for periodic patterns
   - **Generic Stack**: Learnable basis for residual patterns
3. **Doubly Residual Stacking**: 
   - **Backcast**: Reconstructs input to remove explained patterns
   - **Forecast**: Predicts future values
   - Residual passed to next block

### Univariate Approach
- Each time series is modeled **independently**
- No cross-series information sharing
- Scalable to thousands of series (parallel processing)
- Robust scaler applied per series

## Advantages

- **Interpretable Decomposition**: Separates trend, seasonality, and residual components  
- **Pure Forecasting Architecture**: Designed specifically for time series (no adaptations from other domains)  
- **No Manual Feature Engineering**: Automatically learns basis functions  
- **Double Residual Learning**: Efficiently captures complex patterns through stacking  
- **Univariate Scalability**: Processes multiple series independently without memory overhead

## Limitations

- **No Exogenous Variables**: NBEATS does not support external features (use NBEATSx for exogenous support)  
- **Univariate Only**: Cannot leverage cross-series information or correlations  
- **Fixed Architecture Stacks**: Requires careful tuning of stack types and block counts  
- **Longer Training**: Multiple stacks and basis expansions increase computational cost

## When to Use

**Ideal for:**
- Univariate time series with **clear trend and seasonal patterns**
- Datasets with **hundreds to thousands of independent series**
- When **interpretability** of forecast components is important
- Medium to long sequences (10-100+ timesteps)
- When cross-series relationships are not relevant

**Not recommended for:**
- Series requiring exogenous variables (use NBEATSx instead)
- Very short sequences (<6 timesteps)
- When multivariate dependencies are critical

## Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `dropout_prob_theta` | 0.5 | Dropout probability applied to the MLP layers that compute basis coefficients (regularization to prevent overfitting) |
| `max_steps` | 500 | Maximum number of training steps (epochs) for model optimization |
| `n_harmonics` | 2 | Number of Fourier harmonic terms used in the seasonality stack to capture periodic patterns |
| `n_polynomials` (n_basis) | 2 | Degree of polynomial basis functions used in the trend stack to model long-term trends |
| `n_blocks` | [1, 1, 1] | Number of blocks per stack (trend, seasonality, generic) – controls model depth and capacity |
| `mlp_units` | [[512, 512], [512, 512], [512, 512]] | Hidden layer sizes for each stack's MLP – defines the width of fully connected layers processing the input |
| `input_size` | 6 | Number of historical timesteps used as input (lookback window) |
| `h` | 3 | Forecast horizon (number of future timesteps to predict) |
| `scaler_type` | 'robust' | Type of scaler for input normalization (robust scaler is less sensitive to outliers) |
| `loss` | MAE | Loss function for training (Mean Absolute Error for robust forecasting) |

### Stack Configuration
- **Stack 1 (Trend)**: Uses `n_polynomials` polynomial basis functions
- **Stack 2 (Seasonality)**: Uses `n_harmonics` Fourier basis functions
- **Stack 3 (Generic)**: Uses learnable basis functions

### MLP Architecture
Each stack has an MLP defined by `mlp_units`:
- Example: `[[512, 512], [512, 512], [512, 512]]` creates 3 stacks
- Each inner list defines hidden layers for that stack's MLP
- `[512, 512]` = 2 hidden layers with 512 units each





## Model

```python
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses import MAE

model = NBEATS(
    input_size=6,
    h=3,
    loss=MAE(),
    dropout_prob_theta=0.5,
    max_steps=500,
    n_harmonics=2,
    n_polynomials=2,
    n_blocks=[1, 1, 1],
    mlp_units=[[512, 512], [512, 512], [512, 512]],
    scaler_type='robust',
    random_seed=42
)
```

### Data Format
NeuralForecast expects data in long format:
- `unique_id`: Time series identifier
- `ds`: Date/timestamp column
- `y`: Target variable (values to forecast)

### Forecasting Process
1. Each series is scaled independently using robust scaler
2. Model processes series in parallel (batched by unique_id)
3. Forecasts are generated for all series simultaneously
4. Predictions are inverse-transformed to original scale

## Model Results without Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout Prob Theta | Learning Rate | Max Steps | MLP Size | N Blocks | N Harmonics | N Polynomials | Duration (s) |
|-------|-----------------|------------|-------------------|---------------|-----------|----------|----------|-------------|---------------|--------------|
| 0 | 136.42 | 8 | 0.396 | 0.000162 | 200 | 256 | [1, 1, 1] | 4 | 4 | 3.73 |
| 1 | 137.33 | 16 | 0.137 | 0.000231 | 200 | 512 | [3, 3, 3] | 3 | 1 | 4.73 |
| 2 | 136.12 | 4 | 0.412 | 0.000160 | 100 | 256 | [2, 2, 2] | 2 | 3 | 1.90 |
| 3 | 135.85 | 4 | 0.384 | 0.000295 | 100 | 512 | [1, 1, 1] | 2 | 3 | 1.73 |
| 4 | 136.74 | 4 | 0.017 | 0.009780 | 200 | 512 | [2, 2, 2] | 2 | 4 | 3.75 |

### Best Hyperparameters 

Parameters:
- learning_rate: 0.0002951000444000333
- batch_size: 4
- dropout_prob_theta: 0.3842477403805334
- max_steps: 100
- n_harmonics: 2
- n_polynomials: 3
- n_blocks: [1, 1, 1]
- mlp_size: 512

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/nbeats/fold1/predictions_vs_actuals.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/nbeats/fold2/predictions_vs_actuals.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/nbeats/fold3/predictions_vs_actuals.png)


### Fold Results

| Metric | Fold 1 | Fold 2 | Fold 3 | Average |
|--------|--------|--------|--------|---------|
| MSE | 51,403.67 | 53,023.50 | 42,551.48 | 48,992.89 ± 5,636.91 |
| RMSE | 226.72 | 230.27 | 206.28 | 221.09 ± 12.95 |
| MAE | 100.13 | 98.24 | 86.01 | 94.79 ± 7.67 |
| R² | 0.8981 | 0.8829 | 0.9125 | 0.8978 ± 0.0148 |
| SMAPE | 73.63% | 71.94% | 68.83% | 71.47% ± 2.43% |
| **Average** | **48,992.89** | **221.09** | **94.79** | **0.8978** | **71.47%** |

### SMAPE Distribution accross folds

| SMAPE Range | Percentage | Series Count (avg) |
|-------------|------------|-------------------|
| <10% | 5.1% ± 1.7% | 77 |
| 10-20% | 13.3% ± 1.7% | 200 |
| 20-30% | 12.9% ± 2.0% | 194 |
| 30-40% | 9.8% ± 0.5% | 147 |
| >40% | 58.8% ± 5.6% | 884 |


### Comparison to Baseline Model

The NBEATS univariate model achieves an average SMAPE of 71.47% ± 2.43%, which is **0.79 percentage points lower** than the baseline 3-month rolling average (72.26% ± 7.06%). The NBEATS model demonstrates superior performance with significantly more stable results across folds (standard deviation: 2.43% vs 7.06%) compared to the baseline. The model performs best in Fold 3 (68.83% SMAPE), showcasing its ability to decompose time series into interpretable trend and seasonal components through basis expansion. The consistent improvement over the baseline and lower variance across folds indicates that the NBEATS architecture effectively captures complex temporal patterns through its doubly residual stacking mechanism.